In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import pickle

DATA_PATH = '/content/drive/MyDrive/Churn_Hackathon/data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv'
df = pd.read_csv(DATA_PATH)
print(f"✅ Loaded: {df.shape}")

Mounted at /content/drive
✅ Loaded: (7043, 21)


In [2]:
# Drop customerID (not useful for prediction)
df.drop('customerID', axis=1, inplace=True)

# Fix TotalCharges: convert to numeric (some rows have spaces)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Fill missing TotalCharges with median
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

# Encode target variable: Yes → 1, No → 0
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

print("✅ Cleaning done")
print(f"Missing values remaining: {df.isnull().sum().sum()}")

✅ Cleaning done
Missing values remaining: 0


/tmp/ipykernel_1015/329990317.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)


In [3]:
# Separate binary (Yes/No) and multi-class categorical columns
binary_cols = [col for col in df.select_dtypes('object').columns
               if df[col].nunique() == 2]

multi_cols = [col for col in df.select_dtypes('object').columns
              if df[col].nunique() > 2]

print(f"Binary columns: {binary_cols}")
print(f"Multi-class columns: {multi_cols}")

# Label encode binary columns
le = LabelEncoder()
for col in binary_cols:
    df[col] = le.fit_transform(df[col])

# One-hot encode multi-class columns
df = pd.get_dummies(df, columns=multi_cols, drop_first=True)

print(f"\n✅ Encoding done. New shape: {df.shape}")
df.head(2)

Binary columns: ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
Multi-class columns: ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']

✅ Encoding done. New shape: (7043, 31)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,0,1,0,1,0,1,29.85,29.85,0,...,False,False,False,False,False,False,False,False,True,False
1,1,0,0,0,34,1,0,56.95,1889.50,0,...,False,False,False,False,False,True,False,False,False,True


In [4]:
# Create useful derived features
df['AvgMonthlySpend'] = df['TotalCharges'] / (df['tenure'] + 1)  # +1 to avoid divide by zero
df['TenureGroup'] = pd.cut(df['tenure'], bins=[0, 12, 24, 48, 72],
                            labels=['0-1yr', '1-2yr', '2-4yr', '4+yr'])
df['TenureGroup'] = LabelEncoder().fit_transform(df['TenureGroup'].astype(str))

print("✅ Feature engineering done")
print(f"Total features: {df.shape[1] - 1}")  # -1 for target

✅ Feature engineering done
Total features: 32


In [5]:
# Separate features and target
X = df.drop('Churn', axis=1)
y = df['Churn']

# Split: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale numerical features
scaler = StandardScaler()
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'AvgMonthlySpend']

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])  # use same scaler!

print(f"✅ Split done")
print(f"Training set: {X_train.shape}")
print(f"Test set:     {X_test.shape}")
print(f"Churn rate in train: {y_train.mean()*100:.2f}%")
print(f"Churn rate in test:  {y_test.mean()*100:.2f}%")

✅ Split done
Training set: (5634, 32)
Test set:     (1409, 32)
Churn rate in train: 26.54%
Churn rate in test:  26.54%


In [9]:
PROCESSED_PATH = '/content/drive/MyDrive/Churn_Hackathon/data/processed/'

# Save processed splits
X_train.to_csv(PROCESSED_PATH + 'X_train.csv', index=False)
X_test.to_csv(PROCESSED_PATH + 'X_test.csv', index=False)
y_train.to_csv(PROCESSED_PATH + 'y_train.csv', index=False)
y_test.to_csv(PROCESSED_PATH + 'y_test.csv', index=False)

# Save scaler for later use in prediction
with open(PROCESSED_PATH + 'scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("✅ All processed files saved to Drive")

✅ All processed files saved to Drive
